# Returning Tool Results

This notebook implements the examples given in the [langchain documentation](https://python.langchain.com/v0.2/docs/how_to/tool_results_pass_to_model/)

In [1]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b


tools = [add, multiply]

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(model_name='llama3-70b-8192')
llm_with_tools = llm.bind_tools(tools)

In [3]:
from langchain_core.messages import HumanMessage, ToolMessage

query = "What is 3 * 12? Also, what is 11 + 49?"

messages = [HumanMessage(query)]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply}[tool_call["name"].lower()]
    tool_output = selected_tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))
messages

[HumanMessage(content='What is 3 * 12? Also, what is 11 + 49?'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_f4h8', 'function': {'arguments': '{"a":3,"b":12}', 'name': 'multiply'}, 'type': 'function'}, {'id': 'call_tq7m', 'function': {'arguments': '{"a":11,"b":49}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 1045, 'total_tokens': 1129, 'completion_time': 0.24, 'prompt_time': 0.208147905, 'queue_time': None, 'total_time': 0.448147905}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_7ab5f7e105', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-9da46245-b96c-4006-a19c-a5e25d7ee24c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_f4h8'}, {'name': 'add', 'args': {'a': 11, 'b': 49}, 'id': 'call_tq7m'}]),
 ToolMessage(content='36', tool_call_id='call_f4h8'),
 ToolMessage(content='60', tool_call_id='call_tq7m')]

In [4]:
llm_with_tools.invoke(messages)


AIMessage(content='The answer to 3 * 12 is 36 and the answer to 11 + 49 is 60.', response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 1173, 'total_tokens': 1198, 'completion_time': 0.071428571, 'prompt_time': 0.214220711, 'queue_time': None, 'total_time': 0.285649282}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_87cbfbbc4d', 'finish_reason': 'stop', 'logprobs': None}, id='run-6938b944-2a6b-4140-8993-6f421036a8d5-0')